<a id="top"></a>

# Demo: the description is the tool

```yaml
title: "Demo: the description is the tool"
keywords: tool description, recruitment ad, contract, sparse vs rich vs misleading, selection step, haiku contrast, description-implementation mismatch, teacher demo
estimated duration: 14
```

> **Lesson:** L05. Teacher demo — Demo 2 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L05/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Behavior is **distributional**: run each variant twice so the class sees variance, not a single result. Clear outputs before committing.
>
> **Why the raw Anthropic SDK here, not `potato_llm`?** The course's `potato_llm` seam is **text-only** — its `Message` cannot represent `tool_use` / `tool_result` blocks. L05 registers tools and watches the model *choose* and *call* them, so these demos call the raw SDK directly (exactly as [L04](../L04/L0401_intro.md) introduced). The API key still loads through the config seam (`require_anthropic_key`); we never hard-code it. L05 changes the *client*, not where secrets come from.
>
> The point to land: **same name, same schema, different description → different behavior.** The description is the model's only training signal at inference time for *when* to reach for the tool. We also re-run on **Haiku 4.5** to show the design gap widens on a smaller model.

## Contents

- [1. Setup — one tool, three descriptions](#1-setup--one-tool-three-descriptions)
- [2. Read the three descriptions side by side](#2-read-the-three-descriptions-side-by-side)
- [3. The straightforward prompt (P1)](#3-the-straightforward-prompt-p1)
- [4. The ambiguous prompt (P2) — where the description earns its keep](#4-the-ambiguous-prompt-p2--where-the-description-earns-its-keep)
- [5. The misleading description's delayed cost](#5-the-misleading-descriptions-delayed-cost)
- [6. The gap widens on a smaller model (Haiku)](#6-the-gap-widens-on-a-smaller-model-haiku)
- [7. Takeaways](#7-takeaways)

## 1. Setup — one tool, three descriptions

One implementation: `lookup_user(query)` over a tiny in-memory user table. Three tool *definitions* that differ **only** in the `description` field — sparse, rich, misleading — and a `make_tool(description)` helper to swap between them. Two clients: Sonnet 4.6 (anchor) and Haiku 4.5 (the smaller-model contrast).

In [ ]:
import anthropic

from fluffy_potato_curriculum.common.config import require_anthropic_key

client = anthropic.Anthropic(api_key=require_anthropic_key())
SONNET = "claude-sonnet-4-6"  # course anchor
HAIKU = "claude-haiku-4-5"  # smaller model — the design gap bites harder


# The implementation NEVER changes across the three variants — only the description does.
USERS: dict[str, dict[str, str]] = {
    "alex@example.com": {"name": "Alex Kim", "email": "alex@example.com"},
    "alex_kim": {"name": "Alex Kim", "email": "alex@example.com"},
    "priya@example.com": {"name": "Priya Rao", "email": "priya@example.com"},
}


def lookup_user(query: str) -> dict[str, str]:
    """Return the user record for an email or username, or {} if not found."""
    return USERS.get(query.strip().lower(), {})


SPARSE = "Looks up a user."
RICH = (
    "Look up a single user record by exact email or username. "
    "Accepts one query string: an email (e.g. 'alex@example.com') or a username "
    "(e.g. 'alex_kim'). Returns {'name', 'email'} for that one user, or {} if no exact "
    "match. Do NOT call this to list or summarize users in general, and do not guess a "
    "query — if the user has not named a specific person, ask them who they mean instead."
)
MISLEADING = (
    "Looks up any information about any user — email, billing, preferences, and full "
    "account history. Call this for anything user-related."
)  # the implementation returns only {name, email}; this OVERSTATES it.


def make_tool(description: str) -> dict[str, object]:
    """Build the lookup_user tool definition with a swappable description."""
    return {
        "name": "lookup_user",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "an email or username"}},
            "required": ["query"],
        },
    }


def show_turn(resp) -> None:
    """Print a response's blocks (text, and any tool_use name + args)."""
    print("stop_reason:", resp.stop_reason)
    for b in resp.content:
        if b.type == "tool_use":
            args = ", ".join(f"{k}={v!r}" for k, v in b.input.items())
            print(f"  tool_use  {b.name}({args})")
        else:
            print(f"  text      {getattr(b, 'text', '')!r}")


P1 = "Find me the email for the user named Alex."  # straightforward
P2 = "Tell me about our users."  # ambiguous — the revealing case
print("setup ready")

[↑ Back to top](#top)

## 2. Read the three descriptions side by side

Before any model call, put the three descriptions on screen and read them aloud. Ask the class (rhetorically) to predict which produces the best tool use. The name and schema are **identical** in all three — only this prose changes.

In [ ]:
for label, desc in [("SPARSE", SPARSE), ("RICH", RICH), ("MISLEADING", MISLEADING)]:
    print(f"--- {label} ---")
    print(" ", desc)
    print()

[↑ Back to top](#top)

## 3. The straightforward prompt (P1)

Run P1 against each description. All three usually produce a tool call — but watch the **arguments**: the rich description tends to make the model format the query more precisely (it copies the example format), while sparse leaves it guessing the shape.

In [ ]:
for label, desc in [("SPARSE", SPARSE), ("RICH", RICH), ("MISLEADING", MISLEADING)]:
    resp = client.messages.create(
        model=SONNET,
        max_tokens=300,
        tools=[make_tool(desc)],
        messages=[{"role": "user", "content": P1}],
    )
    print(f"=== P1 / {label} ===")
    show_turn(resp)
    print()

[↑ Back to top](#top)

## 4. The ambiguous prompt (P2) — where the description earns its keep

Run the **ambiguous** P2 against each description. This is where behavior diverges: **sparse** often calls the tool with a garbage query; **rich** usually asks for clarification or refuses (its description says *don't guess*); **misleading** calls the tool expecting rich data. Run each twice — the lesson is the *distribution*, not one run.

In [ ]:
for label, desc in [("SPARSE", SPARSE), ("RICH", RICH), ("MISLEADING", MISLEADING)]:
    print(f"=== P2 / {label} (two runs) ===")
    for _ in range(2):
        resp = client.messages.create(
            model=SONNET,
            max_tokens=300,
            tools=[make_tool(desc)],
            messages=[{"role": "user", "content": P2}],
        )
        show_turn(resp)
        print()

[↑ Back to top](#top)

## 5. The misleading description's delayed cost

The misleading description's damage shows up on the *follow-up*. Run P1 with it, then ask for *billing history* — data the tool never returns. The model often calls the same tool again expecting more, because the description **promised** it. A description that overstates the tool is as harmful as one that says too little.

In [ ]:
tool = make_tool(MISLEADING)
messages: list[dict[str, object]] = [{"role": "user", "content": P1}]
first = client.messages.create(model=SONNET, max_tokens=300, tools=[tool], messages=messages)
print("=== first turn (P1, misleading) ===")
show_turn(first)

# Feed back a thin tool_result, then ask for data the tool can't provide.
use = next((b for b in first.content if b.type == "tool_use"), None)
if use is not None:
    messages.append({"role": "assistant", "content": first.content})
    messages.append(
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": use.id,
                    "content": str(lookup_user(use.input.get("query", ""))),
                }
            ],
        }
    )
    messages.append({"role": "user", "content": "Now show me their billing history."})
    followup = client.messages.create(model=SONNET, max_tokens=300, tools=[tool], messages=messages)
    print("\n=== follow-up: 'show me their billing history' ===")
    show_turn(followup)

[↑ Back to top](#top)

## 6. The gap widens on a smaller model (Haiku)

Re-run the revealing P2 case on **Haiku 4.5**. A smaller model leans harder on the description, so the sparse-vs-rich gap is sharper: sparse Haiku is more likely to fire a garbage call, rich Haiku more likely to follow the *don't guess* instruction. The design error bites harder where the model has less to fall back on.

In [ ]:
for label, desc in [("SPARSE", SPARSE), ("RICH", RICH)]:
    resp = client.messages.create(
        model=HAIKU,
        max_tokens=300,
        tools=[make_tool(desc)],
        messages=[{"role": "user", "content": P2}],
    )
    print(f"=== P2 / {label} / HAIKU ===")
    show_turn(resp)
    print()

[↑ Back to top](#top)

## 7. Takeaways

- **Same tool, same model, same task — only the description changed, and behavior changed dramatically.** The description is the model's only training signal at inference time for *when* to reach for the tool.
- A description is two things at once: a **recruitment ad** (when to reach for it) and a **contract** (what it returns). A mismatch in either dimension cascades into bad calls.
- **Write for the model's selection step, not the human reader.** Code comments are for humans; tool descriptions are for the model.
- An **overstated** description (misleading) is as harmful as a sparse one — match the description to the implementation exactly.
- Descriptions reduce variance, they don't eliminate it — which is why the **schema** ([L0507](L0507_lecture.ipynb)) is the next line of defense. The [L0506 lab](L0506_lab_empty.ipynb) has you rewrite a sparse description into a rich one.

[↑ Back to top](#top)